# MolekylvisualiseringEt molekyl er ikke en tekststreng. Men alt vi har gjort så langt, har vært tekst og tall: SMILES-koder, molare masser, tabeller med deskriptorer. På et tidspunkt må vi se på formen.Det er ikke pynt. Kjemi handler i stor grad om at *strukturen bestemmer egenskapene*, og noen spørsmål kan bare besvares ved å se: Hvor er det aktive setet i et enzym? Hvorfor passer akkurat dette molekylet inn der? Hvilken side av molekylet er polar? Hvordan endrer et protein form når det aktiveres?Vi skal se på to bibliotek som tegner molekyler i tre dimensjoner rett i en notebook: `py3Dmol` og `nglview`. De gjør omtrent det samme, men de er gode på hver sine ting, og forskjellen mellom dem er verdt å forstå.Legg merke til hva slags hjelp de gir deg. Begge er innpakninger rundt store JavaScript-bibliotek som tegner molekyler med maskinvareakselerert grafikk i nettleseren. Det er den andre kategorien fra forrige kapittel: maskineri du aldri kommer til å skrive selv.

```{admonition} Læringsutbytte:class: noteEtter å ha arbeidet med denne delen av emnet, skal du kunne:1. Hente molekyl- og proteinstrukturer fra PubChem og Protein Data Bank.2. Tegne molekyler i tre dimensjoner med `py3Dmol` og velge mellom ulike representasjoner.3. Legge på overflater og potensialkart, og tolke hva de viser.4. Tegne proteiner med `nglview` og velge ut bestemte deler av strukturen med et seleksjonsspråk.5. Visualisere en molekyldynamikksimulering som en animasjon.6. Bygge en 3D-struktur fra en SMILES-kode med RDKit og vise den.7. Vurdere hvilket av bibliotekene som passer til en gitt oppgave.```

## Installasjon```python!pip install py3Dmol nglview rdkit mdtraj````nglview` er en såkalt **widget**. Det betyr at det er levende JavaScript koblet til en kjørende Python-kjerne, ikke et statisk bilde. Det får noen konsekvenser vi kommer tilbake til nederst på sida.Eldre oppskrifter på nett ber deg kjøre `jupyter-nbextension enable nglview --py --sys-prefix` etter installasjon. Det trengs ikke lenger i nyere versjoner. Hvis du finner en slik instruks, er kilden sannsynligvis utdatert på flere punkter.

In [ ]:
!pip install py3Dmol nglview rdkit mdtraj

## Hvor kommer strukturene fra?Begge bibliotekene kan hente strukturer direkte fra to store, åpne databaser:**PubChem** inneholder små molekyler: legemidler, naturstoffer, industrikjemikalier. Hver forbindelse har en **CID** (Compound ID). Du finner den ved å søke på pubchem.ncbi.nlm.nih.gov, eller ved å bruke `pubchempy` slik vi gjorde i forrige kapittel.**Protein Data Bank (PDB)** inneholder eksperimentelt bestemte strukturer av store biomolekyler: proteiner, nukleinsyrer og komplekser av dem. Hver struktur har en firetegns **PDB-ID**, som `1PSN` eller `4HHB`. Du finner dem på rcsb.org.Det er verdt å stoppe litt ved forskjellen. En PDB-struktur er et **måleresultat**, som regel fra røntgendiffraksjon, kryoelektronmikroskopi eller NMR. Den har en oppløsning, den har usikkerhet, og den viser molekylet slik det var i akkurat den krystallen eller den prøven. En struktur du genererer selv fra en SMILES-kode er derimot en **modell**, regnet ut av et kraftfelt. Begge deler er nyttige, men de er ikke samme slags kunnskap.

## Del 1: py3Dmol`py3Dmol` er den enkleste av de to. Du henter en struktur, velger en stil, og viser den.

In [ ]:
import py3Dmolparacetamol = py3Dmol.view(query="cid:1983")     # PubChem CIDparacetamol.setStyle({"stick": {"colorscheme": "cyanCarbon"}})paracetamol.zoomTo()paracetamol.show()

Figuren kan roteres med musa, zoomes med rullehjulet og flyttes med høyre musetast.Stilen settes med `setStyle`, som tar en dictionary. Nøkkelen er representasjonstypen, og verdien er en ny dictionary med innstillinger for den typen. De vanligste typene er:| Type | Viser | Egner seg til ||---|---|---|| `line` | tynne streker | store systemer, rask oversikt || `stick` | pinnemodell | små og mellomstore molekyler || `sphere` | kalottmodell | å vise hvor mye plass molekylet tar || `cartoon` | bånd og piler | proteiner og nukleinsyrer || `cross` | kryss per atom | posisjoner uten bindinger |La oss se den samme forbindelsen med tre ulike representasjoner ved siden av hverandre.

In [ ]:
visning = py3Dmol.view(query="cid:2519", viewergrid=(1, 3), width=750, height=250)visning.setStyle({"line": {}},                              viewer=(0, 0))visning.setStyle({"stick": {"colorscheme": "cyanCarbon"}},  viewer=(0, 1))visning.setStyle({"sphere": {"scale": 0.9}},                viewer=(0, 2))visning.zoomTo()visning.show()

```{admonition} Underveisoppgave: Samme molekyl, tre historier:class: tipSe på de tre framstillingene av koffein ovenfor.1. Hvilken av dem gir deg best oversikt over hvilke atomer som er bundet til hvilke?2. Hvilken sier mest om hvor stor plass molekylet tar i en løsning?3. Hvilken ville du valgt for å forklare til en medstudent hvorfor koffein er et plant molekyl?4. Bytt ut CID-en med et molekyl du har jobba med tidligere i emnet, og gjenta vurderinga.Poenget er at en representasjon aldri er nøytral. Den framhever noe og skjuler noe annet, og valget ditt er et faglig valg.```

### ProteinerFor store biomolekyler er `cartoon` den viktigste representasjonen. Den skjuler alle atomene og viser i stedet ryggraden som bånd, slik at sekundærstrukturen blir synlig: alfaheliksene som spiraler og betaplatene som brede piler.

In [ ]:
rna_polymerase = py3Dmol.view(query="pdb:5IYC", width=600, height=450)rna_polymerase.setStyle({"cartoon": {"color": "spectrum"}})rna_polymerase.zoomTo()rna_polymerase.show()

`spectrum` fargelegger fra den ene enden av kjeden til den andre, slik at du kan følge polypeptidkjeden med øynene.Vi kan også kombinere representasjoner. Her viser vi hemoglobin som bånd, men tegner hemgruppene som pinnemodeller og jernionene som kuler, slik at det blir tydelig hvor oksygenet binder seg.

In [ ]:
hemoglobin = py3Dmol.view(query="pdb:4HHB", width=600, height=450)hemoglobin.setStyle({"cartoon": {"color": "spectrum"}})hemoglobin.addStyle({"resn": "HEM"}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.2}})hemoglobin.addStyle({"resn": "HEM", "elem": "Fe"}, {"sphere": {"radius": 0.8, "color": "orange"}})hemoglobin.zoomTo()hemoglobin.show()

Legg merke til forskjellen mellom `setStyle` og `addStyle`. `setStyle` erstatter alt, mens `addStyle` legger noe oppå det som allerede er der, for et utvalg atomer. Utvalget gjøres med en dictionary: `{"resn": "HEM"}` betyr "alle atomer som hører til en residue med navn HEM".```{admonition} Underveisoppgave: Finn hemet:class: tip1. Hvor mange hemgrupper ser du i hemoglobin? Stemmer det med det du vet om strukturen til hemoglobin?2. Zoom inn på én av dem. Hvilket grunnstoff sitter i midten, og hva er koordinasjonstallet?3. Bytt ut `4HHB` med `1MBN` (myoglobin) og gjenta. Hva er den viktigste strukturelle forskjellen mellom de to proteinene, og hvordan henger den sammen med funksjonen deres?```

### Overflater og potensialkartEn overflate viser hvor molekylet "slutter", altså hvor et annet molekyl ville støtt borti det. `VDW` står for van der Waals-radiene, som overflatemodellen bygger på.Hvis vi fargelegger overflata etter ladningsfordeling, får vi et **potensialkart**. Det vanligste fargevalget er en gradvis overgang fra rødt via hvitt til blått, der rødt er elektronrike og blått er elektronfattige områder.

In [ ]:
elektronkart = py3Dmol.view(query="cid:12180", width=500, height=400)  # metylbutanoatelektronkart.setStyle({"stick": {"colorscheme": "cyanCarbon"}})elektronkart.addSurface("VDW", {"opacity": 0.75,                                "colorscheme": {"gradient": "rwb"}})elektronkart.zoomTo()elektronkart.show()

```{admonition} Les fargene med forsiktighet:class: warningFargeskalaen på et slikt kart er *relativ til molekylet du ser på*, ikke absolutt. Det rødeste området på et upolart molekyl kan være langt mindre elektronrikt enn det blåeste området på et ionisk system.Kartet er dessuten regna ut med en forenkla modell, ikke med kvantekjemi. Det er godt nok til å se hvor de polare områdene ligger, men ikke til å lese av tall.``````{admonition} Underveisoppgave: Polaritet du kan se:class: tipLag potensialkart for følgende og sammenlikn dem:1. Vann (CID 962)2. Metan (CID 297)3. Etanol (CID 702)4. Eddiksyre (CID 176)For hvert molekyl: Hvor er det rødeste området, og hvilket atom sitter det på? Stemmer det med elektronegativitetene du henta fra `mendeleev` i forrige kapittel?Forklar deretter hvorfor metan skiller seg ut fra de tre andre, og knytt det til kokepunktene til de fire stoffene.```

### Fra SMILES til 3DVi avslutta forrige kapittel med å lage en 3D-konformasjon i RDKit og hente den ut som en molblokk. Nå kan vi tegne den.Dette er koblinga mellom de to kapitlene: RDKit *lager* strukturen, `py3Dmol` *viser* den, og de to bibliotekene snakker sammen gjennom et standardformat.

In [ ]:
from rdkit import Chemfrom rdkit.Chem import AllChemdef lag_3d(smiles, seed=42):    molekyl = Chem.AddHs(Chem.MolFromSmiles(smiles))    AllChem.EmbedMolecule(molekyl, randomSeed=seed)    AllChem.MMFFOptimizeMolecule(molekyl)    return Chem.MolToMolBlock(molekyl)blokk = lag_3d("CC(C)Cc1ccc(cc1)C(C)C(=O)O")   # ibuprofenvisning = py3Dmol.view(width=500, height=400)visning.addModel(blokk, "mol")visning.setStyle({"stick": {"colorscheme": "cyanCarbon"}})visning.zoomTo()visning.show()

```{admonition} Underveisoppgave: Er modellen god?:class: tipStrukturen ovenfor er regna ut av et kraftfelt, ikke målt.1. Kjør `lag_3d` med tre ulike verdier for `seed` og vis alle tre. Blir de like? Hva forteller det deg?2. Hent den samme forbindelsen fra PubChem med `query="cid:3672"` og sammenlikn med din egen struktur. Hvor stemmer de, og hvor gjør de det ikke?3. Prøv med et molekyl med en lang, fleksibel kjede, for eksempel en fettsyre. Hvordan endrer svaret på spørsmål 1 seg, og hvorfor?4. Formuler i én setning hva `EmbedMolecule` faktisk gir deg, og hva den *ikke* gir deg.```````{admonition} Løsningsforslag:class: dropdown1. For et lite, ganske stivt molekyl blir konformasjonene nokså like, men ikke identiske. `EmbedMolecule` bruker en tilfeldig startgjetning, og `seed` styrer den.2. PubChem-strukturen er også en beregna konformasjon, ikke en måling. De to strukturene vil ha svært like bindingslengder og vinkler, men kan ha ulike torsjonsvinkler rundt enkeltbindinger.3. For en fleksibel kjede spriker resultatene mye mer. Molekylet har mange lavtliggende konformasjoner med omtrent samme energi, og kraftfeltet finner bare *en* av dem.4. `EmbedMolecule` gir deg **én rimelig konformasjon** med fornuftige bindingslengder og vinkler. Den gir deg *ikke* den mest stabile konformasjonen, og heller ikke fordelinga av konformasjoner som molekylet faktisk har i løsning. Til det trenger du en konformasjonssøking eller en molekyldynamikksimulering.````

## Del 2: nglview`nglview` er tyngre enn `py3Dmol`, men gir deg mer kontroll. Den store forskjellen er **seleksjonsspråket**: et lite tekstspråk for å plukke ut nøyaktig de delene av en struktur du er interessert i. Det er dette som gjør `nglview` til det naturlige valget når du jobber med proteiner.`nglview` er også bygd for **trajektorier**, altså strukturer som endrer seg over tid. Den kommer med en avspiller, og er derfor standardvalget for å se på molekyldynamikksimuleringer.La oss starte med pepsin, et enzym i magesekken som bryter ned proteiner til kortere polypeptider.

In [ ]:
import nglview as nvenzym = nv.show_pdbid("1PSN")enzym.layout.width = "600px"enzym.layout.height = "450px"enzym

Legg merke til at den siste linja bare er variabelnavnet. Widgeten vises fordi det er den siste verdien i cella, på samme måte som en dataframe vises uten `print`.### SeleksjonsspråketHer ligger styrken. En seleksjon er en tekststreng som beskriver et utvalg atomer:| Seleksjon | Betyr ||---|---|| `protein` | alle aminosyrerester || `water` | vannmolekyler || `hetero` | alt som ikke er protein eller nukleinsyre, altså ligander, ioner og kofaktorer || `GLY` | alle glysinrester || `TRP or TYR or PHE` | alle aromatiske rester || `1-50` | rest nummer 1 til 50 || `:A` | kjede A || `backbone` | ryggraden || `sidechainAttached` | sidekjedene |Vi kan legge en halvgjennomsiktig overflate på hele proteinet, eller bare på utvalgte rester.

In [ ]:
enzym_overflate = nv.show_pdbid("1PSN")enzym_overflate.add_surface(selection="protein", opacity=0.3)enzym_overflate.layout.width = "600px"enzym_overflate.layout.height = "450px"enzym_overflate

In [ ]:
# Bare glysinrestene får overflate. Da ser du hvor de ligger i strukturen.enzym_glysin = nv.show_pdbid("1PSN")enzym_glysin.add_surface(selection="GLY", opacity=0.4, color="tomato")enzym_glysin.layout.width = "600px"enzym_glysin.layout.height = "450px"enzym_glysin

Vi kan også vise bestemte sidekjeder som pinnemodell oppå båndene. Her tegner vi tryptofanrestene, som er de største og mest hydrofobe av aminosyrene.

In [ ]:
enzym_trp = nv.show_pdbid("1PSN")enzym_trp.add_licorice("TRP")          # TRP er tryptofanenzym_trp.center(selection="TRP")enzym_trp.layout.width = "600px"enzym_trp.layout.height = "450px"enzym_trp

Hold musepekeren over en rest for å se hvilken det er. Det er en av grunnene til at `nglview` er nyttig når du utforsker en struktur du ikke kjenner fra før.### Bytte ut representasjonen helt`clear_representations` fjerner alt som er tegna, og deretter bygger du opp figuren fra bunnen.

In [ ]:
enzym_kule_pinne = nv.show_pdbid("1PSN")enzym_kule_pinne.clear_representations()enzym_kule_pinne.add_representation("ball+stick", selection="protein")enzym_kule_pinne.layout.width = "600px"enzym_kule_pinne.layout.height = "450px"enzym_kule_pinne

```{admonition} Underveisoppgave: Det aktive setet:class: tip1. Vis pepsin (`1PSN`) og legg på en halvgjennomsiktig overflate over hele proteinet. Roter strukturen til du finner den dype kløfta i overflata. Det er det aktive setet.2. Pepsin er en aspartatprotease, og har to katalytisk aktive asparaginsyrerester i bunnen av kløfta. Vis dem med `add_licorice("ASP")` og se om du finner dem. Hvorfor tror du de ligger akkurat der?3. Pepsin skilles ut som proenzymet **pepsinogen**, og endrer struktur ved den lave pH-en i magesekken. Vis pepsinogen (`3PSG`) med samme oppsett. Hva er den viktigste forskjellen du ser?4. Ut fra strukturen: tror du pepsinogen kan ha samme virkning som pepsin? Begrunn med det du ser.```````{admonition} Løsningsforslag:class: dropdown2. De to asparaginsyrerestene ligger i bunnen av kløfta fordi det er der substratet, altså polypeptidkjeden som skal kuttes, må passere. Katalyse krever at de reaktive gruppene er i kontakt med akkurat den bindinga som skal brytes.3. Pepsinogen har et ekstra segment på omtrent 44 aminosyrer i den ene enden. Det ligger som en propp inne i kløfta og dekker det aktive setet.4. Nei. Så lenge proppen sitter der, kommer ikke substratet fram til de katalytiske restene. Ved lav pH endrer ladningsfordelinga seg, propp-segmentet løsner og blir kutta av, og enzymet blir aktivt. Dette er en generell mekanisme: mange fordøyelsesenzymer og enzymer i blodkoagulasjon skilles ut som inaktive forstadier, slik at de ikke bryter ned vevet de blir laget i.````

### Molekyler fra RDKitOgså `nglview` kan vise et RDKit-molekyl direkte, uten omveien om en molblokk.

In [ ]:
from rdkit import Chemfrom rdkit.Chem import AllChemkoffein = Chem.AddHs(Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C)C(=O)N2C"))AllChem.EmbedMolecule(koffein, randomSeed=42)AllChem.MMFFOptimizeMolecule(koffein)visning = nv.show_rdkit(koffein)visning

### Statiske bilderWidgeten er interaktiv, men noen ganger trenger du et vanlig bilde til en rapport. Da kan du be `nglview` om å rendre figuren slik den står nå.Det må gjøres i **to separate celler**. Rendringa skjer i nettleseren og tar litt tid, så bildet er ikke klart før cella er ferdig.

In [ ]:
enzym_trp.render_image()

In [ ]:
enzym_trp._display_image()

## Del 3: SimuleringerAlt vi har sett på så langt, har vært stillbilder. Men molekyler beveger seg, og i **molekyldynamikk (MD)** simulerer vi nettopp den bevegelsen: vi regner ut kreftene mellom alle atomene, flytter dem et lite tidssteg, og gjentar millioner av ganger.Resultatet er en **trajektorie**: en lang serie med strukturer, én per lagra tidssteg. Med MD kan man for eksempel studere hvordan et transportprotein slipper stoffer gjennom en cellemembran, hvordan et legemiddel binder seg til målproteinet sitt, eller hvordan porestrukturen i et materiale påvirker reaktiviteten.Vi skal ikke kjøre simuleringer her, bare se på hvordan du leser inn og viser en trajektorie noen andre har laget. `nglview` kommer med noen demofiler til akkurat dette formålet.For å lese trajektoriefiler trenger vi et bibliotek som forstår formatene. To vanlige valg er `mdtraj` og `MDAnalysis`. Begge finnes for alle operativsystem.

In [ ]:
import nglview as nvimport mdtraj as mdtrajektorie = md.load(nv.datafiles.TRR, top=nv.datafiles.PDB)print(trajektorie)animasjon = nv.show_mdtraj(trajektorie)animasjon.layout.width = "600px"animasjon.layout.height = "450px"animasjon

Trykk på avspillingsknappen under figuren. Skyvefeltet lar deg gå til et bestemt tidssteg.Det samme kan gjøres med `MDAnalysis`:```pythonimport MDAnalysis as mdafrom MDAnalysis.tests.datafiles import PSF, DCDunivers = mda.Universe(PSF, DCD)protein = univers.select_atoms("protein")animasjon = nv.show_mdanalysis(protein)animasjon``````{admonition} Demodataene ligger i en egen pakke:class: warning`MDAnalysis.tests.datafiles` er *ikke* en del av `MDAnalysis` selv. Den ligger i pakka `MDAnalysisTests`, som du må installere separat med `pip install MDAnalysisTests`.Dette er en klassisk snublestein. Feilmeldinga sier at modulen ikke finnes, og det er lett å tro at man har installert feil bibliotek.``````{admonition} Underveisoppgave: Se på bevegelsen:class: tip1. Kjør animasjonen ovenfor. Hvilke deler av strukturen beveger seg mest, og hvilke ligger nesten stille? Hva tror du forklarer forskjellen?2. Legg på en overflate, og se animasjonen på nytt. Blir det lettere eller vanskeligere å se hva som skjer? Hva sier det om valg av representasjon?3. Hvor mange rammer inneholder trajektorien? Bruk `len(trajektorie)`. Hvis hver ramme er lagra hvert 10. pikosekund, hvor lang tid dekker simuleringa?4. Sammenlikn den tida med tida det tar for et enzym å utføre en katalytisk syklus, som typisk er millisekunder. Hva er den praktiske konsekvensen av det forholdet for hva MD-simuleringer kan og ikke kan si noe om?```

## Hvilket bibliotek bør du velge?De to bibliotekene overlapper mye, men er gode på hver sine ting.| | py3Dmol | nglview ||---|---|---|| Små molekyler | Svært godt | Godt || Proteiner | Godt | Svært godt || Å plukke ut deler av en struktur | Enkle utvalg | Fullt seleksjonsspråk || Trajektorier og animasjon | Begrensa | Bygd for det || Flere figurer side om side | `viewergrid` | Ett vindu om gangen || Potensialkart | Innebygd | Krever mer oppsett || Avhengigheter | Få | Flere, inkludert widget-systemet || Overlever et statisk nettsidebygg | Ja | Ofte ikke |Den siste rada er verdt en forklaring, fordi den bestemmer mye i praksis.`py3Dmol` skriver HTML og JavaScript rett inn i output-cella. Når notebooken lagres, følger figuren med, og den vises også hvis noen leser notebooken som en vanlig nettside uten å kjøre koden.`nglview` er en widget, og trenger en levende Python-kjerne for å tegne noe. Lagrer du notebooken og åpner den et sted uten kjerne, for eksempel på en publisert nettside eller på GitHub, blir figuren ofte bare en tom boks.Det er derfor du ser `py3Dmol`-figurene på denne sida, men må kjøre koden selv for å se `nglview`-figurene.```{admonition} Praktisk regel:class: tipSkal figuren *leses* av noen andre uten at de kjører koden, bruk `py3Dmol`.Skal du *utforske* en struktur selv, eller se på en simulering, bruk `nglview`.```

## Sluttoppgaver```{admonition} Oppgave 1: Bindingsforhold du kan se:class: tipVelg tre molekyler som illustrerer ulike bindingsforhold: ett med bare enkeltbindinger, ett med en dobbeltbinding, og ett aromatisk.1. Vis alle tre som pinnemodeller side om side med `viewergrid`.2. Legg på potensialkart for hvert av dem.3. Skriv en kort tekst der du forklarer forskjellene du ser, knytta til elektronegativitet og elektronfordeling.4. Generer de samme tre molekylene fra SMILES med RDKit og sammenlikn geometrien med strukturene fra PubChem. Er noen av dem systematisk forskjellige? Hvorfor?``````{admonition} Oppgave 2: Et enzym og substratet:class: tipVelg et enzym fra PDB som er løst sammen med et substrat, en substratanalog eller en hemmer. Lysozym med en sukkerkjede (`1HEW`) er et godt utgangspunkt, men velg gjerne noe fra ditt eget fagfelt.1. Vis proteinet som bånd og liganden som pinnemodell.2. Bruk seleksjonsspråket i `nglview` til å vise sidekjedene som ligger nær liganden.3. Legg på en halvgjennomsiktig overflate og vis at liganden ligger i en lomme.4. Lag et statisk bilde og skriv en figurtekst på tre til fire setninger som forklarer hva figuren viser.5. Slå opp den katalytiske mekanismen til enzymet og forklar hvilke av restene du ser som er direkte involvert.``````{admonition} Oppgave 3: Konformasjoner og energi:class: tipButan har en velkjent energiprofil for rotasjon rundt den midterste karbon-karbon-bindinga.1. Bygg butan i RDKit og generer 20 ulike konformasjoner med `AllChem.EmbedMultipleConfs`.2. Beregn energien til hver konformasjon med `AllChem.MMFFOptimizeMoleculeConfs`.3. Lag et histogram over energiene. Hvor mange energiminima ser du?4. Vis den laveste og den høyeste konformasjonen i `py3Dmol` og sammenlikn dem.5. Sammenlikn med den teoretiske energiprofilen for butan fra læreboka. Stemmer forholdet mellom anti og gauche?``````{admonition} Oppgave 4: Din egen figur til en rapport:class: tipLag én figur som du kunne brukt i en labrapport eller en presentasjon. Den skal:1. Vise et molekyl eller protein som er relevant for noe du faktisk har gjort på laben.2. Bruke minst to representasjoner i samme figur.3. Ha en bevisst begrunnelse for hvert valg av representasjon og farge.4. Være lagra som et statisk bilde.Lever figuren sammen med en kort tekst på fem til ti setninger der du begrunner valgene dine, og der du sier hva figuren *ikke* viser. Det siste er like viktig som det første.```